In [ ]:
   import nfl_data_py as nfl
   games = nfl.import_schedules(range(2010, 2026))
   games.head()
   

In [ ]:
   %pip install pandas nfl_data_py

In [ ]:
   %pip install pandas nfl_data_py

In [ ]:
   %pip install nflreadpy pandas pyarrow

In [1]:
   import nflreadpy as nfl
   games = nfl.load_schedules(list(range(2010, 2026))).to_pandas()
   games.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2010_01_MIN_NO,2010,REG,1,2010-09-09,Thursday,20:30,MIN,9,NO,...,NaN,00-0005106,00-0020531,Brett Favre,Drew Brees,Brad Childress,Sean Payton,Terry McAulay,NOR00,Louisiana Superdome
1,2010_01_MIA_BUF,2010,REG,1,2010-09-12,Sunday,13:00,MIA,15,BUF,...,7.0,00-0026197,00-0025479,Chad Henne,Trent Edwards,Tony Sparano,Chan Gailey,Clete Blakeman,BUF00,Ralph Wilson Stadium
2,2010_01_DET_CHI,2010,REG,1,2010-09-12,Sunday,13:00,DET,14,CHI,...,10.0,00-0026498,00-0024226,Matthew Stafford,Jay Cutler,Jim Schwartz,Lovie Smith,Gene Steratore,CHI98,Soldier Field
3,2010_01_IND_HOU,2010,REG,1,2010-09-12,Sunday,13:00,IND,24,HOU,...,NaN,00-0010346,00-0022787,Peyton Manning,Matt Schaub,Jim Caldwell,Gary Kubiak,Ed Hochuli,HOU00,Reliant Stadium
4,2010_01_DEN_JAX,2010,REG,1,2010-09-12,Sunday,13:00,DEN,17,JAX,...,10.0,00-0023541,00-0021231,Kyle Orton,David Garrard,Josh McDaniels,Jack Del Rio,Walt Coleman,JAX00,EverBank Field


In [2]:
   print(games.columns.tolist())

['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [3]:
import pandas as pd

# Regular season only
reg = games[games["game_type"] == "REG"].copy()

# Reshape so each game becomes two rows: one for the home team, one for the away team
home = reg[["season", "week", "home_team", "home_coach", "home_score", "away_score"]].copy()
home.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

away = reg[["season", "week", "away_team", "away_coach", "away_score", "home_score"]].copy()
away.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

tg = pd.concat([home, away]).dropna(subset=["pts_for", "pts_against"])
tg["win"] = (tg["pts_for"] > tg["pts_against"]).astype(int)
tg["loss"] = (tg["pts_for"] < tg["pts_against"]).astype(int)
tg["tie"] = (tg["pts_for"] == tg["pts_against"]).astype(int)
tg["pt_diff"] = tg["pts_for"] - tg["pts_against"]

# One row per coach per team per season
coach_seasons = (
    tg.groupby(["season", "team", "coach"])
      .agg(games=("win", "size"), wins=("win", "sum"), losses=("loss", "sum"),
           ties=("tie", "sum"), pt_diff=("pt_diff", "sum"))
      .reset_index()
)
coach_seasons["win_pct"] = (coach_seasons["wins"] + 0.5 * coach_seasons["ties"]) / coach_seasons["games"]

coach_seasons.sort_values(["season", "team"]).head(20)

,season,team,coach,games,wins,losses,ties,pt_diff,win_pct
0,2010,ARI,Ken Whisenhunt,16,5,11,0,-145,0.3125
1,2010,ATL,Mike Smith,16,13,3,0,126,0.8125
2,2010,BAL,John Harbaugh,16,12,4,0,87,0.7500
3,2010,BUF,Chan Gailey,16,4,12,0,-142,0.2500
4,2010,CAR,John Fox,16,2,14,0,-212,0.1250
5,2010,CHI,Lovie Smith,16,11,5,0,48,0.6875
6,2010,CIN,Marvin Lewis,16,4,12,0,-73,0.2500
7,2010,CLE,Eric Mangini,16,5,11,0,-61,0.3125
8,2010,DAL,Jason Garrett,8,5,3,0,29,0.6250
9,2010,DAL,Wade Phillips,8,1,7,0,-71,0.1250


In [4]:
   multi = coach_seasons.groupby(["season", "team"]).filter(lambda d: len(d) > 1)
   multi.sort_values(["season", "team"])

,season,team,coach,games,wins,losses,ties,pt_diff,win_pct
8,2010,DAL,Jason Garrett,8,5,3,0,29,0.625000
9,2010,DAL,Wade Phillips,8,1,7,0,-71,0.125000
10,2010,DEN,Eric Studesville,4,1,3,0,-50,0.250000
11,2010,DEN,Josh McDaniels,12,3,9,0,-77,0.250000
19,2010,MIN,Brad Childress,10,3,7,0,-54,0.300000
20,2010,MIN,Leslie Frazier,6,3,3,0,-13,0.500000
30,2010,SF,Jim Tomsula,1,1,0,0,31,1.000000
31,2010,SF,Mike Singletary,15,5,10,0,-72,0.333333
50,2011,JAX,Jack Del Rio,11,3,8,0,-62,0.272727
51,2011,JAX,Mel Tucker,5,2,3,0,-24,0.400000


In [5]:
   coach_seasons = coach_seasons.sort_values(["coach", "team", "season"])
   coach_seasons["tenure"] = coach_seasons.groupby(["coach", "team"]).cumcount() + 1

In [1]:
# Last season each coach appeared with each team
last = (coach_seasons.groupby(["coach", "team"])["season"]
        .max().reset_index(name="last_season"))

# Anyone whose last season is before 2025 has left that team
departed = last[last["last_season"] < 2025].copy()
departed["fired"] = 1   # default guess; you'll correct the exceptions
departed = departed.sort_values(["last_season", "team"])

departed.to_csv("departures.csv", index=False)
print(len(departed))
departed.head(20)

NameError: name 'coach_seasons' is not defined

In [2]:
import nflreadpy as nfl
import pandas as pd

games = nfl.load_schedules(list(range(2010, 2026))).to_pandas()

reg = games[games["game_type"] == "REG"].copy()

home = reg[["season", "week", "home_team", "home_coach", "home_score", "away_score"]].copy()
home.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

away = reg[["season", "week", "away_team", "away_coach", "away_score", "home_score"]].copy()
away.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

tg = pd.concat([home, away]).dropna(subset=["pts_for", "pts_against"])
tg["win"] = (tg["pts_for"] > tg["pts_against"]).astype(int)
tg["loss"] = (tg["pts_for"] < tg["pts_against"]).astype(int)
tg["tie"] = (tg["pts_for"] == tg["pts_against"]).astype(int)
tg["pt_diff"] = tg["pts_for"] - tg["pts_against"]

coach_seasons = (
    tg.groupby(["season", "team", "coach"])
      .agg(games=("win", "size"), wins=("win", "sum"), losses=("loss", "sum"),
           ties=("tie", "sum"), pt_diff=("pt_diff", "sum"))
      .reset_index()
)
coach_seasons["win_pct"] = (coach_seasons["wins"] + 0.5 * coach_seasons["ties"]) / coach_seasons["games"]

coach_seasons = coach_seasons.sort_values(["coach", "team", "season"])
coach_seasons["tenure"] = coach_seasons.groupby(["coach", "team"]).cumcount() + 1

coach_seasons.head()

,season,team,coach,games,wins,losses,ties,pt_diff,win_pct,tenure
532,2025,NYJ,Aaron Glenn,17,3,14,0,-203,0.176471,1
90,2012,NO,Aaron Kromer,6,2,4,0,-6,0.333333,1
222,2016,MIA,Adam Gase,16,10,6,0,-17,0.625000,1
255,2017,MIA,Adam Gase,16,6,10,0,-112,0.375000,2
290,2018,MIA,Adam Gase,16,7,9,0,-114,0.437500,3


In [3]:
dep = pd.read_csv("departures.csv")

coach_seasons = coach_seasons.merge(
    dep[["coach", "team", "last_season", "fired"]],
    left_on=["coach", "team", "season"],
    right_on=["coach", "team", "last_season"],
    how="left"
)
coach_seasons["fired"] = coach_seasons["fired"].fillna(0).astype(int)
coach_seasons.drop(columns="last_season", inplace=True)

coach_seasons["fired"].value_counts()

FileNotFoundError: [Errno 2] No such file or directory: 'departures.csv'

In [4]:
last = (coach_seasons.groupby(["coach", "team"])["season"]
        .max().reset_index(name="last_season"))

departed = last[last["last_season"] < 2025].copy()
departed["fired"] = 1
departed = departed.sort_values(["last_season", "team"])

departed.to_csv("departures.csv", index=False)
print(len(departed))
departed.head(20)


131


,coach,team,last_season,fired
76,John Fox,CAR,2010,1
43,Eric Mangini,CLE,2010,1
160,Wade Phillips,DAL,2010,1
44,Eric Studesville,DEN,2010,1
83,Josh McDaniels,DEN,2010,1
15,Brad Childress,MIN,2010,1
153,Tom Cable,OAK,2010,1
115,Mike Singletary,SF,2010,1
63,Jeff Fisher,TEN,2010,1
67,Jim Caldwell,IND,2011,1


In [5]:
dep = pd.read_csv("departures.csv")

coach_seasons = coach_seasons.merge(
    dep[["coach", "team", "last_season", "fired"]],
    left_on=["coach", "team", "season"],
    right_on=["coach", "team", "last_season"],
    how="left"
)
coach_seasons["fired"] = coach_seasons["fired"].fillna(0).astype(int)
coach_seasons.drop(columns="last_season", inplace=True)

coach_seasons["fired"].value_counts()

fired
0    409
1    131
Name: count, dtype: int64

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

# Drop 2025 (firings after that season aren't labeled yet) and tiny interim stints
df = coach_seasons[(coach_seasons["season"] < 2025) & (coach_seasons["games"] >= 4)].copy()
df["pt_diff_pg"] = df["pt_diff"] / df["games"]

features = ["win_pct", "pt_diff_pg", "tenure"]

# Time-based split: train on older seasons, test on newer ones
train = df[df["season"] <= 2019]
test = df[df["season"] >= 2020].copy()

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(train[features], train["fired"])

test["fire_prob"] = model.predict_proba(test[features])[:, 1]

print("Test AUC:", round(roc_auc_score(test["fired"], test["fire_prob"]), 3))
print(dict(zip(features, model.named_steps["logisticregression"].coef_[0].round(2))))

test.sort_values("fire_prob", ascending=False)[
    ["season", "team", "coach", "wins", "losses", "pt_diff", "fire_prob", "fired"]
].head(15)

ModuleNotFoundError: No module named 'sklearn'

In [7]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (9.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 22.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 50.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

# Drop 2025 (firings after that season aren't labeled yet) and tiny interim stints
df = coach_seasons[(coach_seasons["season"] < 2025) & (coach_seasons["games"] >= 4)].copy()
df["pt_diff_pg"] = df["pt_diff"] / df["games"]

features = ["win_pct", "pt_diff_pg", "tenure"]

# Time-based split: train on older seasons, test on newer ones
train = df[df["season"] <= 2019]
test = df[df["season"] >= 2020].copy()

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(train[features], train["fired"])

test["fire_prob"] = model.predict_proba(test[features])[:, 1]

print("Test AUC:", round(roc_auc_score(test["fired"], test["fire_prob"]), 3))
print(dict(zip(features, model.named_steps["logisticregression"].coef_[0].round(2))))

test.sort_values("fire_prob", ascending=False)[
    ["season", "team", "coach", "wins", "losses", "pt_diff", "fire_prob", "fired"]
].head(15)


Test AUC: 0.858
{'win_pct': np.float64(-0.97), 'pt_diff_pg': np.float64(-0.53), 'tenure': np.float64(0.2)}


,season,team,coach,wins,losses,pt_diff,fire_prob,fired
57,2020,HOU,Bill O'Brien,0,4,-46,0.898311,1
104,2020,ATL,Dan Quinn,0,5,-39,0.855902,1
130,2020,JAX,Doug Marrone,1,15,-186,0.845910,1
49,2023,NE,Bill Belichick,4,13,-130,0.788353,1
6,2020,NYJ,Adam Gase,2,14,-214,0.781415,1
141,2023,CAR,Frank Reich,1,10,-119,0.757817,1
270,2024,CLE,Kevin Stefanski,3,14,-177,0.738562,0
194,2022,IND,Jeff Saturday,1,7,-87,0.726512,1
525,2021,JAX,Urban Meyer,2,11,-160,0.721881,1
107,2020,DET,Darrell Bevell,1,4,-66,0.689987,1


In [1]:
import nflreadpy as nfl
import pandas as pd

games = nfl.load_schedules(list(range(2010, 2026))).to_pandas()

reg = games[games["game_type"] == "REG"].copy()

home = reg[["season", "week", "home_team", "home_coach", "home_score", "away_score"]].copy()
home.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

away = reg[["season", "week", "away_team", "away_coach", "away_score", "home_score"]].copy()
away.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

tg = pd.concat([home, away]).dropna(subset=["pts_for", "pts_against"])
tg["win"] = (tg["pts_for"] > tg["pts_against"]).astype(int)
tg["loss"] = (tg["pts_for"] < tg["pts_against"]).astype(int)
tg["tie"] = (tg["pts_for"] == tg["pts_against"]).astype(int)
tg["pt_diff"] = tg["pts_for"] - tg["pts_against"]

coach_seasons = (
    tg.groupby(["season", "team", "coach"])
      .agg(games=("win", "size"), wins=("win", "sum"), losses=("loss", "sum"),
           ties=("tie", "sum"), pt_diff=("pt_diff", "sum"))
      .reset_index()
)
coach_seasons["win_pct"] = (coach_seasons["wins"] + 0.5 * coach_seasons["ties"]) / coach_seasons["games"]

coach_seasons = coach_seasons.sort_values(["coach", "team", "season"])
coach_seasons["tenure"] = coach_seasons.groupby(["coach", "team"]).cumcount() + 1

coach_seasons.head()

,season,team,coach,games,wins,losses,ties,pt_diff,win_pct,tenure
532,2025,NYJ,Aaron Glenn,17,3,14,0,-203,0.176471,1
90,2012,NO,Aaron Kromer,6,2,4,0,-6,0.333333,1
222,2016,MIA,Adam Gase,16,10,6,0,-17,0.625000,1
255,2017,MIA,Adam Gase,16,6,10,0,-112,0.375000,2
290,2018,MIA,Adam Gase,16,7,9,0,-114,0.437500,3


In [2]:
dep = pd.read_csv("departures.csv")

coach_seasons = coach_seasons.merge(
    dep[["coach", "team", "last_season", "fired"]],
    left_on=["coach", "team", "season"],
    right_on=["coach", "team", "last_season"],
    how="left"
)
coach_seasons["fired"] = coach_seasons["fired"].fillna(0).astype(int)
coach_seasons.drop(columns="last_season", inplace=True)

coach_seasons["fired"].value_counts()



fired
0    409
1    131
Name: count, dtype: int64

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

# Drop 2025 (firings after that season aren't labeled yet) and tiny interim stints
df = coach_seasons[(coach_seasons["season"] < 2025) & (coach_seasons["games"] >= 4)].copy()
df["pt_diff_pg"] = df["pt_diff"] / df["games"]

features = ["win_pct", "pt_diff_pg", "tenure"]

# Time-based split: train on older seasons, test on newer ones
train = df[df["season"] <= 2019]
test = df[df["season"] >= 2020].copy()

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(train[features], train["fired"])

test["fire_prob"] = model.predict_proba(test[features])[:, 1]

print("Test AUC:", round(roc_auc_score(test["fired"], test["fire_prob"]), 3))
print(dict(zip(features, model.named_steps["logisticregression"].coef_[0].round(2))))

test.sort_values("fire_prob", ascending=False)[
    ["season", "team", "coach", "wins", "losses", "pt_diff", "fire_prob", "fired"]
].head(15)


Test AUC: 0.858
{'win_pct': np.float64(-0.97), 'pt_diff_pg': np.float64(-0.53), 'tenure': np.float64(0.2)}


,season,team,coach,wins,losses,pt_diff,fire_prob,fired
57,2020,HOU,Bill O'Brien,0,4,-46,0.898311,1
104,2020,ATL,Dan Quinn,0,5,-39,0.855902,1
130,2020,JAX,Doug Marrone,1,15,-186,0.845910,1
49,2023,NE,Bill Belichick,4,13,-130,0.788353,1
6,2020,NYJ,Adam Gase,2,14,-214,0.781415,1
141,2023,CAR,Frank Reich,1,10,-119,0.757817,1
270,2024,CLE,Kevin Stefanski,3,14,-177,0.738562,0
194,2022,IND,Jeff Saturday,1,7,-87,0.726512,1
525,2021,JAX,Urban Meyer,2,11,-160,0.721881,1
107,2020,DET,Darrell Bevell,1,4,-66,0.689987,1


In [4]:
import pandas as pd
test["bucket"] = pd.qcut(test["fire_prob"], 5, duplicates="drop")
test.groupby("bucket", observed=True).agg(
    avg_predicted=("fire_prob", "mean"),
    actual_rate=("fired", "mean"),
    n=("fired", "size"))

,avg_predicted,actual_rate,n
bucket,,,
"(0.009999999999999998, 0.0639]",0.038035,0.029412,34
"(0.0639, 0.134]",0.093935,0.029412,34
"(0.134, 0.27]",0.197214,0.181818,33
"(0.27, 0.451]",0.355685,0.411765,34
"(0.451, 0.898]",0.630874,0.647059,34


In [5]:
cols = ["season", "team", "coach", "games", "wins", "losses", "pt_diff", "tenure", "fire_prob", "fired"]

print("HIGH risk but NOT fired (false alarms):")
display(test[test["fired"] == 0].sort_values("fire_prob", ascending=False)[cols].head(15))

print("Actually FIRED but model rated low (misses):")
display(test[test["fired"] == 1].sort_values("fire_prob", ascending=True)[cols].head(15))

HIGH risk but NOT fired (false alarms):


,season,team,coach,games,wins,losses,pt_diff,tenure,fire_prob,fired
270,2024,CLE,Kevin Stefanski,17,3,14,-177,5,0.738562,0
66,2024,NYG,Brian Daboll,17,3,14,-142,3,0.665902,0
62,2024,TEN,Brian Callahan,17,3,14,-149,1,0.633387,0
307,2022,CHI,Matt Eberflus,17,3,14,-137,1,0.619625,0
444,2021,NYJ,Robert Saleh,17,4,13,-194,1,0.616950,0
94,2021,DET,Dan Campbell,17,3,13,-142,1,0.590759,0
109,2024,CAR,Dave Canales,17,5,12,-193,1,0.545127,0
250,2023,ARI,Jonathan Gannon,17,4,13,-125,1,0.534549,0
478,2022,LA,Sean McVay,17,5,12,-77,6,0.513881,0
534,2020,CIN,Zac Taylor,16,4,11,-113,2,0.493929,0


Actually FIRED but model rated low (misses):


,season,team,coach,games,wins,losses,pt_diff,tenure,fire_prob,fired
79,2021,TB,Bruce Arians,17,13,4,158,3,0.024412,1
247,2021,LV,Jon Gruden,5,3,2,-7,2,0.111902,1
503,2022,CAR,Steve Wilks,12,6,6,2,1,0.142290,1
442,2021,LV,Rich Bisaccia,12,7,5,-58,1,0.142874,1
70,2021,MIA,Brian Flores,17,9,8,-32,3,0.168853,1
530,2021,DEN,Vic Fangio,17,7,10,13,3,0.225673,1
495,2021,NO,Sean Payton,17,9,8,29,11,0.234372,1
400,2021,MIN,Mike Zimmer,17,8,9,-1,8,0.266610,1
27,2020,LAC,Anthony Lynn,16,7,9,-42,4,0.271105,1
32,2023,ATL,Arthur Smith,17,7,10,-52,3,0.286119,1


In [6]:
dep = pd.read_csv("departures.csv")

set_to_zero = [
    ("Gary Kubiak","DEN"), ("Bruce Arians","ARI"), ("Sean Payton","NO"), ("Doug Marrone","BUF"),
    ("Jeff Fisher","STL"), ("Jon Gruden","OAK"),
    ("Eric Studesville","DEN"), ("Mel Tucker","JAX"), ("Todd Bowles","MIA"), ("Aaron Kromer","NO"),
    ("Joe Vitt","NO"), ("Wade Phillips","HOU"), ("Tony Sparano","OAK"), ("Pat Shurmur","PHI"),
    ("Anthony Lynn","BUF"), ("Steve Spagnuolo","NYG"), ("Gregg Williams","CLE"), ("Joe Philbin","GB"),
    ("Bill Callahan","WAS"), ("Darrell Bevell","DET"), ("Romeo Crennel","HOU"), ("Darrell Bevell","JAX"),
    ("Rich Bisaccia","LV"), ("Steve Wilks","CAR"), ("Jay Rosburg","DEN"), ("Jeff Saturday","IND"),
    ("Chris Tabor","CAR"), ("Giff Smith","LAC"),
]

mask = dep.set_index(["coach", "team"]).index.isin(set_to_zero)
dep.loc[mask, "fired"] = 0
dep.to_csv("departures.csv", index=False)
print(mask.sum(), "rows changed")
dep["fired"].value_counts()

28 rows changed


fired
1    103
0     28
Name: count, dtype: int64

In [7]:
import nflreadpy as nfl
import pandas as pd

games = nfl.load_schedules(list(range(2010, 2026))).to_pandas()

reg = games[games["game_type"] == "REG"].copy()

home = reg[["season", "week", "home_team", "home_coach", "home_score", "away_score"]].copy()
home.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

away = reg[["season", "week", "away_team", "away_coach", "away_score", "home_score"]].copy()
away.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

tg = pd.concat([home, away]).dropna(subset=["pts_for", "pts_against"])
tg["win"] = (tg["pts_for"] > tg["pts_against"]).astype(int)
tg["loss"] = (tg["pts_for"] < tg["pts_against"]).astype(int)
tg["tie"] = (tg["pts_for"] == tg["pts_against"]).astype(int)
tg["pt_diff"] = tg["pts_for"] - tg["pts_against"]

coach_seasons = (
    tg.groupby(["season", "team", "coach"])
      .agg(games=("win", "size"), wins=("win", "sum"), losses=("loss", "sum"),
           ties=("tie", "sum"), pt_diff=("pt_diff", "sum"))
      .reset_index()
)
coach_seasons["win_pct"] = (coach_seasons["wins"] + 0.5 * coach_seasons["ties"]) / coach_seasons["games"]

coach_seasons = coach_seasons.sort_values(["coach", "team", "season"])
coach_seasons["tenure"] = coach_seasons.groupby(["coach", "team"]).cumcount() + 1

coach_seasons.head()


,season,team,coach,games,wins,losses,ties,pt_diff,win_pct,tenure
532,2025,NYJ,Aaron Glenn,17,3,14,0,-203,0.176471,1
90,2012,NO,Aaron Kromer,6,2,4,0,-6,0.333333,1
222,2016,MIA,Adam Gase,16,10,6,0,-17,0.625000,1
255,2017,MIA,Adam Gase,16,6,10,0,-112,0.375000,2
290,2018,MIA,Adam Gase,16,7,9,0,-114,0.437500,3


In [8]:
dep = pd.read_csv("departures.csv")

coach_seasons = coach_seasons.merge(
    dep[["coach", "team", "last_season", "fired"]],
    left_on=["coach", "team", "season"],
    right_on=["coach", "team", "last_season"],
    how="left"
)
coach_seasons["fired"] = coach_seasons["fired"].fillna(0).astype(int)
coach_seasons.drop(columns="last_season", inplace=True)

coach_seasons["fired"].value_counts()


fired
0    437
1    103
Name: count, dtype: int64

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

# Drop 2025 (firings after that season aren't labeled yet) and tiny interim stints
df = coach_seasons[(coach_seasons["season"] < 2025) & (coach_seasons["games"] >= 4)].copy()
df["pt_diff_pg"] = df["pt_diff"] / df["games"]

features = ["win_pct", "pt_diff_pg", "tenure"]

# Time-based split: train on older seasons, test on newer ones
train = df[df["season"] <= 2019]
test = df[df["season"] >= 2020].copy()

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(train[features], train["fired"])

test["fire_prob"] = model.predict_proba(test[features])[:, 1]

print("Test AUC:", round(roc_auc_score(test["fired"], test["fire_prob"]), 3))
print(dict(zip(features, model.named_steps["logisticregression"].coef_[0].round(2))))

test.sort_values("fire_prob", ascending=False)[
    ["season", "team", "coach", "wins", "losses", "pt_diff", "fire_prob", "fired"]
].head(15)


Test AUC: 0.833
{'win_pct': np.float64(-1.43), 'pt_diff_pg': np.float64(-0.18), 'tenure': np.float64(0.4)}


,season,team,coach,wins,losses,pt_diff,fire_prob,fired
57,2020,HOU,Bill O'Brien,0,4,-46,0.935769,1
104,2020,ATL,Dan Quinn,0,5,-39,0.916540,1
49,2023,NE,Bill Belichick,4,13,-130,0.889236,1
130,2020,JAX,Doug Marrone,1,15,-186,0.866942,1
270,2024,CLE,Kevin Stefanski,3,14,-177,0.732506,0
6,2020,NYJ,Adam Gase,2,14,-214,0.719146,1
141,2023,CAR,Frank Reich,1,10,-119,0.718965,1
194,2022,IND,Jeff Saturday,1,7,-87,0.666444,0
66,2024,NYG,Brian Daboll,3,14,-142,0.644727,0
525,2021,JAX,Urban Meyer,2,11,-160,0.627819,1


In [10]:
cols = ["season", "team", "coach", "games", "wins", "losses", "pt_diff", "tenure", "fire_prob", "fired"]

print("HIGH risk but NOT fired (false alarms):")
display(test[test["fired"] == 0].sort_values("fire_prob", ascending=False)[cols].head(15))

print("Actually FIRED but model rated low (misses):")
display(test[test["fired"] == 1].sort_values("fire_prob", ascending=True)[cols].head(15))


HIGH risk but NOT fired (false alarms):


,season,team,coach,games,wins,losses,pt_diff,tenure,fire_prob,fired
270,2024,CLE,Kevin Stefanski,17,3,14,-177,5,0.732506,0
194,2022,IND,Jeff Saturday,8,1,7,-87,1,0.666444,0
66,2024,NYG,Brian Daboll,17,3,14,-142,3,0.644727,0
87,2023,CAR,Chris Tabor,6,1,5,-61,1,0.590884,0
62,2024,TEN,Brian Callahan,17,3,14,-149,1,0.563572,0
307,2022,CHI,Matt Eberflus,17,3,14,-137,1,0.558565,0
107,2020,DET,Darrell Bevell,5,1,4,-66,1,0.552730,0
427,2021,SEA,Pete Carroll,17,7,10,29,12,0.543231,0
478,2022,LA,Sean McVay,17,5,12,-77,6,0.538745,0
94,2021,DET,Dan Campbell,17,3,13,-142,1,0.507284,0


Actually FIRED but model rated low (misses):


,season,team,coach,games,wins,losses,pt_diff,tenure,fire_prob,fired
79,2021,TB,Bruce Arians,17,13,4,158,3,0.014689,1
247,2021,LV,Jon Gruden,5,3,2,-7,2,0.053570,1
70,2021,MIA,Brian Flores,17,9,8,-32,3,0.102796,1
530,2021,DEN,Vic Fangio,17,7,10,13,3,0.200351,1
27,2020,LAC,Anthony Lynn,16,7,9,-42,4,0.214464,1
32,2023,ATL,Arthur Smith,17,7,10,-52,3,0.218581,1
255,2023,LV,Josh McDaniels,8,3,5,-61,2,0.259199,1
400,2021,MIN,Mike Zimmer,17,8,9,-1,8,0.287122,1
61,2023,LAC,Brandon Staley,14,5,9,-42,3,0.293806,1
323,2020,DET,Matt Patricia,11,4,7,-76,3,0.307517,1


In [11]:
base_features = ["win_pct", "pt_diff_pg", "tenure"]
new_features = ["win_pct", "pt_diff_pg", "tenure", "vs_exp_pg", "win_pct_prev"]

for name, feats in [("Baseline", base_features), ("With expectations", new_features)]:
    m = make_pipeline(StandardScaler(), LogisticRegression())
    m.fit(train[feats], train["fired"])
    p = m.predict_proba(test[feats])[:, 1]
    print(f"{name}: AUC = {roc_auc_score(test['fired'], p):.3f}")

Baseline: AUC = 0.833


KeyError: "['vs_exp_pg', 'win_pct_prev'] not in index"

In [1]:
import numpy as np

In [2]:
base_features = ["win_pct", "pt_diff_pg", "tenure"]

m = make_pipeline(StandardScaler(), LogisticRegression())
m.fit(train[base_features], train["fired"])

weights = pd.DataFrame({
    "feature": base_features,
    "weight": m.named_steps["logisticregression"].coef_[0],
})
weights["odds_multiplier"] = np.exp(weights["weight"])
print(weights.round(2))
print("baseline (intercept):", m.named_steps["logisticregression"].intercept_.round(2))


NameError: name 'make_pipeline' is not defined

In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

df = coach_seasons[(coach_seasons["season"] < 2025) & (coach_seasons["games"] >= 4)].copy()
df["pt_diff_pg"] = df["pt_diff"] / df["games"]

train = df[df["season"] <= 2019]
test = df[df["season"] >= 2020].copy()

base_features = ["win_pct", "pt_diff_pg", "tenure"]

m = make_pipeline(StandardScaler(), LogisticRegression())
m.fit(train[base_features], train["fired"])

weights = pd.DataFrame({
    "feature": base_features,
    "weight": m.named_steps["logisticregression"].coef_[0],
})
weights["odds_multiplier"] = np.exp(weights["weight"])
print(weights.round(2))
print("baseline (intercept):", m.named_steps["logisticregression"].intercept_.round(2))

NameError: name 'coach_seasons' is not defined

In [4]:
import nflreadpy as nfl
import pandas as pd

games = nfl.load_schedules(list(range(2010, 2026))).to_pandas()

reg = games[games["game_type"] == "REG"].copy()

home = reg[["season", "week", "home_team", "home_coach", "home_score", "away_score"]].copy()
home.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

away = reg[["season", "week", "away_team", "away_coach", "away_score", "home_score"]].copy()
away.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

tg = pd.concat([home, away]).dropna(subset=["pts_for", "pts_against"])
tg["win"] = (tg["pts_for"] > tg["pts_against"]).astype(int)
tg["loss"] = (tg["pts_for"] < tg["pts_against"]).astype(int)
tg["tie"] = (tg["pts_for"] == tg["pts_against"]).astype(int)
tg["pt_diff"] = tg["pts_for"] - tg["pts_against"]

coach_seasons = (
    tg.groupby(["season", "team", "coach"])
      .agg(games=("win", "size"), wins=("win", "sum"), losses=("loss", "sum"),
           ties=("tie", "sum"), pt_diff=("pt_diff", "sum"))
      .reset_index()
)
coach_seasons["win_pct"] = (coach_seasons["wins"] + 0.5 * coach_seasons["ties"]) / coach_seasons["games"]

coach_seasons = coach_seasons.sort_values(["coach", "team", "season"])
coach_seasons["tenure"] = coach_seasons.groupby(["coach", "team"]).cumcount() + 1

coach_seasons.head()

,season,team,coach,games,wins,losses,ties,pt_diff,win_pct,tenure
532,2025,NYJ,Aaron Glenn,17,3,14,0,-203,0.176471,1
90,2012,NO,Aaron Kromer,6,2,4,0,-6,0.333333,1
222,2016,MIA,Adam Gase,16,10,6,0,-17,0.625000,1
255,2017,MIA,Adam Gase,16,6,10,0,-112,0.375000,2
290,2018,MIA,Adam Gase,16,7,9,0,-114,0.437500,3


In [5]:
dep = pd.read_csv("departures.csv")

coach_seasons = coach_seasons.merge(
    dep[["coach", "team", "last_season", "fired"]],
    left_on=["coach", "team", "season"],
    right_on=["coach", "team", "last_season"],
    how="left"
)
coach_seasons["fired"] = coach_seasons["fired"].fillna(0).astype(int)
coach_seasons.drop(columns="last_season", inplace=True)

coach_seasons["fired"].value_counts()

fired
0    410
1    130
Name: count, dtype: int64

In [6]:
# Drop these first so rerunning the cell doesn't create duplicate columns
coach_seasons = coach_seasons.drop(columns=["vs_exp_pg", "win_pct_prev"], errors="ignore")

# How much each team beat (or missed) the point spread, per game
g = games[games["game_type"] == "REG"].dropna(subset=["spread_line", "result"]).copy()
g["home_vs_exp"] = g["result"] - g["spread_line"]

h = g[["season", "home_team", "home_coach", "home_vs_exp"]].copy()
h.columns = ["season", "team", "coach", "vs_exp"]

a = g[["season", "away_team", "away_coach", "home_vs_exp"]].copy()
a.columns = ["season", "team", "coach", "vs_exp"]
a["vs_exp"] = -a["vs_exp"]   # flip sign for the away team

exp = (pd.concat([h, a])
         .groupby(["season", "team", "coach"])["vs_exp"].mean()
         .reset_index(name="vs_exp_pg"))

coach_seasons = coach_seasons.merge(exp, on=["season", "team", "coach"], how="left")

# Last season's win % for the same coach and team
coach_seasons = coach_seasons.sort_values(["coach", "team", "season"])
coach_seasons["win_pct_prev"] = (coach_seasons.groupby(["coach", "team"])["win_pct"]
                                 .shift(1).fillna(0.5))
coach_seasons["vs_exp_pg"] = coach_seasons["vs_exp_pg"].fillna(0)

coach_seasons[["season", "team", "coach", "win_pct", "vs_exp_pg", "win_pct_prev"]].tail()

,season,team,coach,win_pct,vs_exp_pg,win_pct_prev
535,2021,CIN,Zac Taylor,0.588235,4.470588,0.281250
536,2022,CIN,Zac Taylor,0.750000,1.875000,0.588235
537,2023,CIN,Zac Taylor,0.529412,-1.764706,0.750000
538,2024,CIN,Zac Taylor,0.529412,-0.970588,0.529412
539,2025,CIN,Zac Taylor,0.352941,-2.382353,0.529412


In [7]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

df = coach_seasons[(coach_seasons["season"] < 2025) & (coach_seasons["games"] >= 4)].copy()
df["pt_diff_pg"] = df["pt_diff"] / df["games"]

train = df[df["season"] <= 2019]
test = df[df["season"] >= 2020].copy()

base_features = ["win_pct", "pt_diff_pg", "tenure"]

m = make_pipeline(StandardScaler(), LogisticRegression())
m.fit(train[base_features], train["fired"])

weights = pd.DataFrame({
    "feature": base_features,
    "weight": m.named_steps["logisticregression"].coef_[0],
})
weights["odds_multiplier"] = np.exp(weights["weight"])
print(weights.round(2))
print("baseline (intercept):", m.named_steps["logisticregression"].intercept_.round(2))

      feature  weight  odds_multiplier
0     win_pct   -0.97             0.38
1  pt_diff_pg   -0.53             0.59
2      tenure    0.20             1.22
baseline (intercept): [-1.56]


In [8]:
for test_year in range(2016, 2025):
    tr = df[df["season"] < test_year]
    te = df[df["season"] == test_year]
    if te["fired"].sum() == 0:
        continue
    results = []
    for feats in (base_features, new_features):
        m = make_pipeline(StandardScaler(), LogisticRegression()).fit(tr[feats], tr["fired"])
        results.append(roc_auc_score(te["fired"], m.predict_proba(te[feats])[:, 1]))
    print(test_year, "baseline:", round(results[0], 3), "new:", round(results[1], 3))

NameError: name 'new_features' is not defined

In [9]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

# Rebuild df so it includes the new columns
df = coach_seasons[(coach_seasons["season"] < 2025) & (coach_seasons["games"] >= 4)].copy()
df["pt_diff_pg"] = df["pt_diff"] / df["games"]

train = df[df["season"] <= 2019]
test = df[df["season"] >= 2020].copy()

base_features = ["win_pct", "pt_diff_pg", "tenure"]
new_features = ["win_pct", "pt_diff_pg", "tenure", "vs_exp_pg", "win_pct_prev"]

# Single train/test split
for name, feats in [("Baseline", base_features), ("With expectations", new_features)]:
    m = make_pipeline(StandardScaler(), LogisticRegression())
    m.fit(train[feats], train["fired"])
    p = m.predict_proba(test[feats])[:, 1]
    print(f"{name}: AUC = {roc_auc_score(test['fired'], p):.3f}")

Baseline: AUC = 0.877
With expectations: AUC = 0.887


In [10]:
for test_year in range(2016, 2025):
    tr = df[df["season"] < test_year]
    te = df[df["season"] == test_year]
    if te["fired"].sum() == 0:
        continue
    results = []
    for feats in (base_features, new_features):
        m = make_pipeline(StandardScaler(), LogisticRegression()).fit(tr[feats], tr["fired"])
        results.append(roc_auc_score(te["fired"], m.predict_proba(te[feats])[:, 1]))
    print(test_year, "baseline:", round(results[0], 3), "new:", round(results[1], 3))

2016 baseline: 0.776 new: 0.769
2017 baseline: 0.745 new: 0.775
2018 baseline: 0.792 new: 0.842
2019 baseline: 0.824 new: 0.868
2020 baseline: 0.936 new: 0.983
2021 baseline: 0.771 new: 0.735
2022 baseline: 0.884 new: 0.878
2023 baseline: 0.947 new: 0.987
2024 baseline: 0.823 new: 0.811


In [11]:
new_features = ["win_pct", "pt_diff_pg", "tenure", "vs_exp_pg", "win_pct_prev"]

m = make_pipeline(StandardScaler(), LogisticRegression())
m.fit(train[new_features], train["fired"])
test["fire_prob"] = m.predict_proba(test[new_features])[:, 1]

print("Pooled test AUC:", round(roc_auc_score(test["fired"], test["fire_prob"]), 3))

test["bucket"] = pd.qcut(test["fire_prob"], 5, duplicates="drop")
test.groupby("bucket", observed=True).agg(
    avg_predicted=("fire_prob", "mean"),
    actual_rate=("fired", "mean"),
    n=("fired", "size"))

Pooled test AUC: 0.887


,avg_predicted,actual_rate,n
bucket,,,
"(0.0046, 0.0566]",0.032065,0.000000,34
"(0.0566, 0.13]",0.091114,0.088235,34
"(0.13, 0.261]",0.188851,0.060606,33
"(0.261, 0.429]",0.338428,0.382353,34
"(0.429, 0.919]",0.633283,0.735294,34


In [12]:
import nflreadpy as nfl
import pandas as pd

games = nfl.load_schedules(list(range(2010, 2027))).to_pandas()

reg = games[games["game_type"] == "REG"].copy()

home = reg[["season", "week", "home_team", "home_coach", "home_score", "away_score"]].copy()
home.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

away = reg[["season", "week", "away_team", "away_coach", "away_score", "home_score"]].copy()
away.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

tg = pd.concat([home, away]).dropna(subset=["pts_for", "pts_against"])
tg["win"] = (tg["pts_for"] > tg["pts_against"]).astype(int)
tg["loss"] = (tg["pts_for"] < tg["pts_against"]).astype(int)
tg["tie"] = (tg["pts_for"] == tg["pts_against"]).astype(int)
tg["pt_diff"] = tg["pts_for"] - tg["pts_against"]

coach_seasons = (
    tg.groupby(["season", "team", "coach"])
      .agg(games=("win", "size"), wins=("win", "sum"), losses=("loss", "sum"),
           ties=("tie", "sum"), pt_diff=("pt_diff", "sum"))
      .reset_index()
)
coach_seasons["win_pct"] = (coach_seasons["wins"] + 0.5 * coach_seasons["ties"]) / coach_seasons["games"]

coach_seasons = coach_seasons.sort_values(["coach", "team", "season"])
coach_seasons["tenure"] = coach_seasons.groupby(["coach", "team"]).cumcount() + 1

coach_seasons.head()

,season,team,coach,games,wins,losses,ties,pt_diff,win_pct,tenure
532,2025,NYJ,Aaron Glenn,17,3,14,0,-203.0,0.176471,1
564,2026,NYJ,Aaron Glenn,6,2,4,0,-10.0,0.333333,2
90,2012,NO,Aaron Kromer,6,2,4,0,-6.0,0.333333,1
222,2016,MIA,Adam Gase,16,10,6,0,-17.0,0.625000,1
255,2017,MIA,Adam Gase,16,6,10,0,-112.0,0.375000,2


In [13]:
final_features = ["win_pct", "pt_diff_pg", "tenure", "vs_exp_pg", "win_pct_prev"]

coach_seasons["pt_diff_pg"] = coach_seasons["pt_diff"] / coach_seasons["games"]

# Train on every labeled season (2010-2024)
train_all = coach_seasons[(coach_seasons["season"] < 2025) & (coach_seasons["games"] >= 4)]
final_model = make_pipeline(StandardScaler(), LogisticRegression())
final_model.fit(train_all[final_features], train_all["fired"])

# Score 2026, keeping only the coach who coached each team's most recent game
now = coach_seasons[coach_seasons["season"] == 2026].copy()
latest = (tg[tg["season"] == 2026].sort_values("week")
            .groupby("team").tail(1)[["team", "coach"]])
now = now.merge(latest, on=["team", "coach"])

now["hot_seat_pct"] = (final_model.predict_proba(now[final_features])[:, 1] * 100).round(1)
now["record"] = now["wins"].astype(int).astype(str) + "-" + now["losses"].astype(int).astype(str)

board = now.sort_values("hot_seat_pct", ascending=False)[
    ["coach", "team", "record", "pt_diff", "vs_exp_pg", "tenure", "hot_seat_pct"]]
board.head(15)

KeyError: "['vs_exp_pg', 'win_pct_prev'] not in index"

In [14]:
import nflreadpy as nfl
import pandas as pd

games = nfl.load_schedules(list(range(2010, 2027))).to_pandas()

reg = games[games["game_type"] == "REG"].copy()

home = reg[["season", "week", "home_team", "home_coach", "home_score", "away_score"]].copy()
home.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

away = reg[["season", "week", "away_team", "away_coach", "away_score", "home_score"]].copy()
away.columns = ["season", "week", "team", "coach", "pts_for", "pts_against"]

tg = pd.concat([home, away]).dropna(subset=["pts_for", "pts_against"])
tg["win"] = (tg["pts_for"] > tg["pts_against"]).astype(int)
tg["loss"] = (tg["pts_for"] < tg["pts_against"]).astype(int)
tg["tie"] = (tg["pts_for"] == tg["pts_against"]).astype(int)
tg["pt_diff"] = tg["pts_for"] - tg["pts_against"]

coach_seasons = (
    tg.groupby(["season", "team", "coach"])
      .agg(games=("win", "size"), wins=("win", "sum"), losses=("loss", "sum"),
           ties=("tie", "sum"), pt_diff=("pt_diff", "sum"))
      .reset_index()
)
coach_seasons["win_pct"] = (coach_seasons["wins"] + 0.5 * coach_seasons["ties"]) / coach_seasons["games"]

coach_seasons = coach_seasons.sort_values(["coach", "team", "season"])
coach_seasons["tenure"] = coach_seasons.groupby(["coach", "team"]).cumcount() + 1

coach_seasons.head()

,season,team,coach,games,wins,losses,ties,pt_diff,win_pct,tenure
532,2025,NYJ,Aaron Glenn,17,3,14,0,-203.0,0.176471,1
564,2026,NYJ,Aaron Glenn,6,2,4,0,-10.0,0.333333,2
90,2012,NO,Aaron Kromer,6,2,4,0,-6.0,0.333333,1
222,2016,MIA,Adam Gase,16,10,6,0,-17.0,0.625000,1
255,2017,MIA,Adam Gase,16,6,10,0,-112.0,0.375000,2


In [15]:
dep = pd.read_csv("departures.csv")

coach_seasons = coach_seasons.merge(
    dep[["coach", "team", "last_season", "fired"]],
    left_on=["coach", "team", "season"],
    right_on=["coach", "team", "last_season"],
    how="left"
)
coach_seasons["fired"] = coach_seasons["fired"].fillna(0).astype(int)
coach_seasons.drop(columns="last_season", inplace=True)

coach_seasons["fired"].value_counts()

fired
0    442
1    130
Name: count, dtype: int64

In [16]:
# Drop these first so rerunning doesn't create duplicate columns
coach_seasons = coach_seasons.drop(columns=["vs_exp_pg", "win_pct_prev"], errors="ignore")

# How much each team beat (or missed) the point spread, per game
g = games[games["game_type"] == "REG"].dropna(subset=["spread_line", "result"]).copy()
g["home_vs_exp"] = g["result"] - g["spread_line"]

h = g[["season", "home_team", "home_coach", "home_vs_exp"]].copy()
h.columns = ["season", "team", "coach", "vs_exp"]

a = g[["season", "away_team", "away_coach", "home_vs_exp"]].copy()
a.columns = ["season", "team", "coach", "vs_exp"]
a["vs_exp"] = -a["vs_exp"]   # flip sign for the away team

exp = (pd.concat([h, a])
         .groupby(["season", "team", "coach"])["vs_exp"].mean()
         .reset_index(name="vs_exp_pg"))

coach_seasons = coach_seasons.merge(exp, on=["season", "team", "coach"], how="left")

# Last season's win % for the same coach and team
coach_seasons = coach_seasons.sort_values(["coach", "team", "season"])
coach_seasons["win_pct_prev"] = (coach_seasons.groupby(["coach", "team"])["win_pct"]
                                 .shift(1).fillna(0.5))
coach_seasons["vs_exp_pg"] = coach_seasons["vs_exp_pg"].fillna(0)

coach_seasons[["season", "team", "coach", "win_pct", "vs_exp_pg", "win_pct_prev"]].tail()

,season,team,coach,win_pct,vs_exp_pg,win_pct_prev
567,2022,CIN,Zac Taylor,0.750000,1.875000,0.588235
568,2023,CIN,Zac Taylor,0.529412,-1.764706,0.750000
569,2024,CIN,Zac Taylor,0.529412,-0.970588,0.529412
570,2025,CIN,Zac Taylor,0.352941,-2.382353,0.529412
571,2026,CIN,Zac Taylor,0.600000,1.100000,0.352941


In [17]:
final_features = ["win_pct", "pt_diff_pg", "tenure", "vs_exp_pg", "win_pct_prev"]

coach_seasons["pt_diff_pg"] = coach_seasons["pt_diff"] / coach_seasons["games"]

# Train on every labeled season (2010-2024)
train_all = coach_seasons[(coach_seasons["season"] < 2025) & (coach_seasons["games"] >= 4)]
final_model = make_pipeline(StandardScaler(), LogisticRegression())
final_model.fit(train_all[final_features], train_all["fired"])

# Score 2026, keeping only the coach who coached each team's most recent game
now = coach_seasons[coach_seasons["season"] == 2026].copy()
latest = (tg[tg["season"] == 2026].sort_values("week")
            .groupby("team").tail(1)[["team", "coach"]])
now = now.merge(latest, on=["team", "coach"])

now["hot_seat_pct"] = (final_model.predict_proba(now[final_features])[:, 1] * 100).round(1)
now["record"] = now["wins"].astype(int).astype(str) + "-" + now["losses"].astype(int).astype(str)

board = now.sort_values("hot_seat_pct", ascending=False)[
    ["coach", "team", "record", "pt_diff", "vs_exp_pg", "tenure", "hot_seat_pct"]]
board.head(15)

,coach,team,record,pt_diff,vs_exp_pg,tenure,hot_seat_pct
10,Jim Harbaugh,LAC,0-6,-61.0,-8.500000,3,89.2
8,Jeff Hafley,MIA,0-5,-62.0,-3.900000,1,81.6
12,Jonathan Gannon,ARI,2-4,-52.0,-2.833333,4,67.9
5,Dan Quinn,WAS,1-4,-40.0,-2.250000,3,64.5
29,Todd Bowles,TB,1-5,-19.0,-1.250000,5,60.2
24,Robert Saleh,TEN,1-5,-45.0,-2.083333,1,56.5
22,Nick Sirianni,PHI,2-4,-31.0,-5.583333,6,53.0
26,Sean McVay,LA,2-4,-3.0,-5.166667,10,52.8
13,Kellen Moore,NO,2-4,-26.0,-2.416667,2,49.0
28,Shane Steichen,IND,2-4,-20.0,-2.916667,4,47.0


In [18]:
print(coach_seasons.columns.tolist())

['season', 'team', 'coach', 'games', 'wins', 'losses', 'ties', 'pt_diff', 'win_pct', 'tenure', 'fired', 'vs_exp_pg', 'win_pct_prev', 'pt_diff_pg']


In [19]:
from datetime import date

full_board = now.sort_values("hot_seat_pct", ascending=False)[
    ["coach", "team", "record", "pt_diff", "vs_exp_pg", "tenure", "hot_seat_pct"]].copy()
full_board["vs_exp_pg"] = full_board["vs_exp_pg"].round(1)
full_board["updated"] = str(date.today())

full_board.to_csv("hot_seat.csv", index=False)
print(len(full_board), "coaches saved")

32 coaches saved


In [20]:
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Hot Seat", page_icon="🔥", layout="wide")

df = pd.read_csv("hot_seat.csv")

def tier(p):
    if p >= 60: return "🔥 On Fire"
    if p >= 30: return "🌶️ Hot"
    if p >= 10: return "🟡 Warm"
    return "🧊 Cold"

df["status"] = df["hot_seat_pct"].apply(tier)

st.title("🔥 Hot Seat")
st.caption("A statistical estimate of how close each NFL head coach is to being fired. "
           f"Last updated {df['updated'].iloc[0]}.")

# Top 5 spotlight
cols = st.columns(5)
for col, (_, row) in zip(cols, df.head(5).iterrows()):
    col.metric(f"{row['coach']} ({row['team']})", f"{row['hot_seat_pct']:.0f}%", row["record"])

st.subheader("Full leaderboard")
st.dataframe(
    df[["status", "coach", "team", "record", "pt_diff", "vs_exp_pg", "tenure", "hot_seat_pct"]],
    column_config={
        "status": "Status",
        "coach": "Coach",
        "team": "Team",
        "record": "Record",
        "pt_diff": "Point diff",
        "vs_exp_pg": "vs. spread / game",
        "tenure": "Seasons w/ team",
        "hot_seat_pct": st.column_config.ProgressColumn(
            "Fire probability", min_value=0, max_value=100, format="%.0f%%"),
    },
    hide_index=True,
    use_container_width=True,
)

with st.expander("How it works"):
    st.write(
        "The model is a logistic regression trained on NFL coach seasons from 2010 to 2024. "
        "It uses win percentage, point differential, tenure, performance against the betting "
        "spread, and last season's record. It can't see contract buyouts, owner patience, or "
        "off-field issues, so treat it as a ranking of statistical risk, not a prediction "
        "of any team's decision."
    )

ModuleNotFoundError: No module named 'streamlit'

In [21]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 30.5 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 22.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 28.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 32.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [streamlit]20 [streamlit]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Hot Seat", page_icon="🔥", layout="wide")

df = pd.read_csv("hot_seat.csv")

def tier(p):
    if p >= 60: return "🔥 On Fire"
    if p >= 30: return "🌶️ Hot"
    if p >= 10: return "🟡 Warm"
    return "🧊 Cold"

df["status"] = df["hot_seat_pct"].apply(tier)

st.title("🔥 Hot Seat")
st.caption("A statistical estimate of how close each NFL head coach is to being fired. "
           f"Last updated {df['updated'].iloc[0]}.")

# Top 5 spotlight
cols = st.columns(5)
for col, (_, row) in zip(cols, df.head(5).iterrows()):
    col.metric(f"{row['coach']} ({row['team']})", f"{row['hot_seat_pct']:.0f}%", row["record"])

st.subheader("Full leaderboard")
st.dataframe(
    df[["status", "coach", "team", "record", "pt_diff", "vs_exp_pg", "tenure", "hot_seat_pct"]],
    column_config={
        "status": "Status",
        "coach": "Coach",
        "team": "Team",
        "record": "Record",
        "pt_diff": "Point diff",
        "vs_exp_pg": "vs. spread / game",
        "tenure": "Seasons w/ team",
        "hot_seat_pct": st.column_config.ProgressColumn(
            "Fire probability", min_value=0, max_value=100, format="%.0f%%"),
    },
    hide_index=True,
    use_container_width=True,
)

with st.expander("How it works"):
    st.write(
        "The model is a logistic regression trained on NFL coach seasons from 2010 to 2024. "
        "It uses win percentage, point differential, tenure, performance against the betting "
        "spread, and last season's record. It can't see contract buyouts, owner patience, or "
        "off-field issues, so treat it as a ranking of statistical risk, not a prediction "
        "of any team's decision."
    )

2026-10-22 22:14:01.489 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-22 22:14:01.491 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-22 22:14:01.512 
  command:

    streamlit run /Users/loganfeather/Python/HotSeat/.venv/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-10-22 22:14:01.512 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-22 22:14:01.513 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-22 22:14:01.513 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-22 22:14:01.513 Thread 'MainThread': missing ScriptRunContext! This warni

In [23]:
streamlit run app.py

SyntaxError: invalid syntax (3737097518.py, line 1)